# Export MIDI-DDSP Decoder to ONNX

Exports the pretrained MIDI-DDSP synthesis decoder to ONNX.

**Run all cells in order.** The ONNX file downloads at the end.

In [ ]:
# 1. Install runtime deps (NOT midi-ddsp — its setup.py is broken)
!pip install -q tf2onnx onnx onnxruntime ddsp pretty_midi note-seq
print('Deps installed.')

In [ ]:
# 2. Clone midi-ddsp source and add to path (bypass broken pip install)
import os, sys
if not os.path.exists('midi-ddsp'):
    !git clone --depth 1 https://github.com/magenta/midi-ddsp.git
sys.path.insert(0, 'midi-ddsp')

# Verify import works
from midi_ddsp.utils.midi_synthesis_utils import load_pretrained_model
print('midi_ddsp imported successfully (from source).')

In [ ]:
# 3. Download pretrained weights
import zipfile
from urllib.request import urlretrieve

WEIGHTS_URL = 'https://github.com/magenta/midi-ddsp/raw/models/midi_ddsp_model_weights_urmp_9_10.zip'
WEIGHTS_DIR = 'midi-ddsp/midi_ddsp'
ZIP_PATH = os.path.join(WEIGHTS_DIR, 'weights.zip')

if not os.path.exists(os.path.join(WEIGHTS_DIR, 'midi_ddsp_model_weights_urmp_9_10')):
    print('Downloading weights (~46 MB)...')
    urlretrieve(WEIGHTS_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(WEIGHTS_DIR)
    os.remove(ZIP_PATH)
    print('Done.')
else:
    print('Weights already downloaded.')

In [ ]:
# 4. Load the pretrained synthesis generator
synthesis_generator, expression_generator = load_pretrained_model()
print(f'Loaded: {type(synthesis_generator).__name__}')

# Show the call signature
import inspect
if hasattr(synthesis_generator, 'call'):
    print(f'call signature: {inspect.signature(synthesis_generator.call)}')

In [ ]:
# 5. Find the correct call pattern
import tensorflow as tf
import numpy as np

N = 250  # 1 second at 250 Hz frame rate

test_inputs = [
    ('3D', [
        tf.constant(np.full((1, N, 1), 440.0, dtype=np.float32)),
        tf.constant(np.full((1, N, 1), -30.0, dtype=np.float32)),
        tf.constant([[0]], dtype=tf.int32),
    ]),
    ('2D', [
        tf.constant(np.full((1, N), 440.0, dtype=np.float32)),
        tf.constant(np.full((1, N), -30.0, dtype=np.float32)),
        tf.constant([0], dtype=tf.int32),
    ]),
]

working = None
for name, args in test_inputs:
    try:
        out = synthesis_generator(*args, training=False)
        keys = list(out.keys()) if isinstance(out, dict) else 'not dict'
        print(f'✓ {name} works. Keys: {keys}')
        if isinstance(out, dict):
            for k, v in out.items():
                print(f'  {k}: {v.shape}')
        working = (name, args)
        break
    except Exception as e:
        print(f'✗ {name}: {str(e)[:120]}')

if not working:
    raise RuntimeError('No call pattern worked')

In [ ]:
# 6. Export to SavedModel
SAVED_DIR = '/tmp/ddsp_saved'
name, args = working

if name == '3D':
    @tf.function(input_signature=[
        tf.TensorSpec([1, None, 1], tf.float32, name='f0_hz'),
        tf.TensorSpec([1, None, 1], tf.float32, name='loudness_db'),
        tf.TensorSpec([1, 1], tf.int32, name='instrument_id'),
    ])
    def export_fn(f0, ld, inst):
        out = synthesis_generator(f0, ld, inst, training=False)
        return {k: v for k, v in out.items() if 'amplitude' in k or 'harmonic' in k or 'noise' in k}
else:
    @tf.function(input_signature=[
        tf.TensorSpec([1, None], tf.float32, name='f0_hz'),
        tf.TensorSpec([1, None], tf.float32, name='loudness_db'),
        tf.TensorSpec([1], tf.int32, name='instrument_id'),
    ])
    def export_fn(f0, ld, inst):
        out = synthesis_generator(f0, ld, inst, training=False)
        return {k: v for k, v in out.items() if 'amplitude' in k or 'harmonic' in k or 'noise' in k}

cf = export_fn.get_concrete_function(*args)
tf.saved_model.save(synthesis_generator, SAVED_DIR, signatures={'serving_default': cf})
print(f'SavedModel → {SAVED_DIR}')

In [ ]:
# 7. Convert to ONNX
import tf2onnx, onnx

ONNX_PATH = '/tmp/ddsp_decoder.onnx'
tf2onnx.convert.from_saved_model(SAVED_DIR, output_path=ONNX_PATH)

model = onnx.load(ONNX_PATH)
onnx.checker.check_model(model)
size = os.path.getsize(ONNX_PATH)
print(f'\nONNX: {size / 1024 / 1024:.1f} MB')
print(f'Inputs: {[i.name for i in model.graph.input]}')
print(f'Outputs: {[o.name for o in model.graph.output]}')

In [ ]:
# 8. Validate
import onnxruntime as ort

sess = ort.InferenceSession(ONNX_PATH)
feed = {}
for inp in sess.get_inputs():
    shape = [d if isinstance(d, int) else 250 for d in inp.shape]
    if 'f0' in inp.name: feed[inp.name] = np.full(shape, 440.0, dtype=np.float32)
    elif 'loud' in inp.name or 'ld' in inp.name: feed[inp.name] = np.full(shape, -30.0, dtype=np.float32)
    elif 'inst' in inp.name: feed[inp.name] = np.zeros(shape, dtype=np.int32)
    else: feed[inp.name] = np.zeros(shape, dtype=np.float32)

result = sess.run(None, feed)
for i, o in enumerate(sess.get_outputs()):
    print(f'{o.name}: shape={result[i].shape}, range=[{result[i].min():.4f}, {result[i].max():.4f}]')

ok = all(np.abs(r).max() > 1e-6 for r in result)
print(f'\n{"✓ PASSED" if ok else "✗ FAILED"}')

In [ ]:
# 9. Download
from google.colab import files
files.download(ONNX_PATH)
print(f'\nUpload: hf upload jcosta33/vocoder-models ddsp_decoder.onnx ddsp-decoder/ddsp_decoder.onnx --repo-type model')
print(f'Update DDSP_MODEL_SIZE_BYTES = {size}')